# `regionprops` benchmark — Julia side

Interactive companion to `benchmarks/benchmark_regionprops.jl`. Same fixture and
pinned algorithm; swap the property list and re-time single calls without
re-running the harness.

**Prerequisite.** Generate the fixtures once:

```
julia --project=benchmarks -t 1 benchmarks/export_labels.jl
```

**Scope.** Only the six properties with a like-for-like scikit-image
counterpart are studied, and `convex_area_algorithm` is pinned to
`PixelConvexArea()` throughout — `PolygonConvexArea` is excluded from the
benchmark (it returns a continuous polygon area over pixel centres, not
commensurable with a pixel count; see `issue_polygon_solidity.md`).

Plots live in the Python comparison notebook; the `benchmarks` environment has
no plotting dependency, so this one stays tabular.

> **Kernel: Julia 1.12 required.** The next cell checks and stops with
> instructions otherwise. On Julia 1.11 the dependencies fail to precompile
> (`PrecompileTools` hits `UndefVarError: StaticData not defined in Base`,
> which is 1.12-only), after several minutes of apparently-normal output.
>
> In **VS Code** the Julia extension supplies its own kernel and ignores the
> Jupyter kernelspec, so it is the usual way to end up on the wrong version.

In [1]:
# Kernel check. Run this first.
#
# IceFloeTracker requires Julia 1.12 (`julia = "1.12"` in Project.toml) and
# benchmarks/Manifest.toml is resolved for it. On an older kernel the failure
# is a slow, confusing cascade -- PrecompileTools hits
# `UndefVarError: StaticData not defined in Base` (1.12-only), which takes
# PrettyTables, DataFrames and IceFloeTracker down with it after several
# minutes of precompiling. Fail immediately and say what to do instead.
if VERSION < v"1.12"
    error("""
    This notebook needs Julia 1.12; this kernel is $(VERSION).

    In Jupyter:  Kernel > Change Kernel... > "Julia 1.12".
    In VS Code:  the Julia extension supplies its own kernel and ignores the
                 Jupyter kernelspec. Set `julia.executablePath` to a 1.12
                 binary, or pick "Julia 1.12" in the kernel selector.

    If the Jupyter kernel is missing or broken:
        julia -e 'using Pkg; Pkg.add("IJulia"); using IJulia; installkernel("Julia")'
    """)
end
println("Julia ", VERSION, "  ", Sys.BINDIR)

Julia 1

.12.7  /home/cpaniaguam/.julia/juliaup/julia-1.12.7+0.x64.linux.gnu/bin


In [2]:
using Pkg

# Locate benchmarks/Project.toml from the notebook's directory, falling back to
# a short upward search so the notebook still works if it is moved.
function benchmarks_dir()
    d = @__DIR__
    for _ in 1:4
        isfile(joinpath(d, "Project.toml")) && occursin("benchmarks", d) && return d
        d = dirname(d)
    end
    error("could not locate the benchmarks environment from $(@__DIR__)")
end

const BENCH = benchmarks_dir()
Pkg.activate(BENCH)
BENCH

  Activating 

project at `~/IceFloeTracker.jl/benchmarks`


"/home/cpaniaguam/IceFloeTracker.jl/benchmarks"

In [3]:
using BenchmarkTools
using DataFrames
using SHA: sha256
using Printf: @printf
using IceFloeTracker: regionprops_table, PixelConvexArea

const FIXTURE_DIR = joinpath(BENCH, "fixtures")

# The six properties that map 1:1 onto scikit-image.
const PROPERTIES = [
    :area, :perimeter, :convex_area, :solidity, :major_axis_length, :minor_axis_length
]

# Pinned everywhere, so `:solidity` is always the pixel-counted ratio.
const ALG = PixelConvexArea()

[ Info: Precompiling IceFloeTracker [04643c7a-9ac6-48c5-822f-2704f9e70bd3](cache misses: wrong dep version loaded (6), dep missing source (3))


[ Info: Precompiling IceFloeTracker [04643c7a-9ac6-48c5-822f-2704f9e70bd3] (cache misses: wrong dep version loaded (12), dep missing source (6))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up



SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up



SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


(::PixelConvexArea) (generic function with 2 methods)

## 1. Load a fixture

The digest is verified before anything else: both languages must read the same
bytes, or the comparison means nothing. The full scene is 19.3 Mpx with 1910 floes; a `convex_area` call takes a few
seconds.

In [4]:
function read_sidecar(path)
    d = Dict{String,Any}()
    for line in eachline(path)
        m = match(r"\"(\w+)\":\s*(.+?),?\s*$", line)
        isnothing(m) && continue
        k, raw = m.captures[1], m.captures[2]
        d[k] = startswith(raw, "\"") ? String(strip(raw, ['"'])) : parse(Int, raw)
    end
    return d
end

"""Read a fixture written by export_labels.jl, verifying its SHA-256."""
function load_fixture(name)
    binpath = joinpath(FIXTURE_DIR, "labels_$name.bin")
    isfile(binpath) || error("missing $binpath -- run export_labels.jl first")
    meta = read_sidecar(joinpath(FIXTURE_DIR, "labels_$name.json"))

    digest = bytes2hex(open(sha256, binpath))
    digest == meta["sha256"] ||
        error("fixture $name digest mismatch -- regenerate with export_labels.jl")

    rows, cols = meta["rows"], meta["cols"]
    buf = Vector{Int32}(undef, rows * cols)
    read!(binpath, buf)
    # Stored C-ordered for NumPy; Julia is column-major, hence the permute.
    return Matrix{Int64}(permutedims(reshape(ltoh.(buf), cols, rows))), meta
end

labels, meta = load_fixture("full")
@printf("%s: %d x %d (%.1f Mpx), %d labels\n",
        meta["name"], meta["rows"], meta["cols"],
        meta["rows"] * meta["cols"] / 1e6, meta["n_labels"])

full: 5680 x 3392 (19.3 Mpx), 1910 labels


## 2. The values

`regionprops_table` returns a `DataFrame`. Note the row count: `regionprops_table` keeps labels with
`area > minimum_area` — *strictly* greater, default 1 — so single-pixel regions
are dropped. scikit-image keeps them, which is why the Python side applies the
same filter before any comparison.

In [5]:
props = regionprops_table(labels; properties=vcat(:label, PROPERTIES),
                          minimum_area=1, convex_area_algorithm=ALG)
println("rows: ", nrow(props), "   (labels in fixture: ", meta["n_labels"], ")")
first(props, 8)

rows: 1834

   (labels in fixture: 1910)


Row,area,convex_area,label,major_axis_length,minor_axis_length,perimeter,solidity
,Int64,Float64,Int64,Any,Any,Float64,Float64
1,790,899.0,1,35.6249,31.2925,150.711,0.878754
2,11478,13169.0,2,162.246,105.641,652.002,0.871592
3,407,443.0,3,35.1078,16.7976,98.7696,0.918736
4,261,272.0,4,21.8759,15.5181,64.8701,0.959559
5,256,265.0,5,22.6918,15.3907,66.2843,0.966038
6,170,178.0,6,16.3219,13.8532,51.799,0.955056
7,327,339.0,7,27.1807,15.8607,74.5269,0.964602
8,285,292.0,8,28.3289,13.2285,72.7696,0.976027


In [6]:
# Distribution of each property, and how many are non-finite.
# `:convex_area`/`:solidity` are NaN below 4 px and where the hull degenerates.
summary_rows = map(PROPERTIES) do p
    v = collect(skipmissing(props[!, p]))
    finite = filter(isfinite, v)
    (property = p,
     n = length(v),
     nonfinite = length(v) - length(finite),
     min = isempty(finite) ? NaN : minimum(finite),
     median = isempty(finite) ? NaN : sort(finite)[cld(length(finite), 2)],
     max = isempty(finite) ? NaN : maximum(finite))
end
DataFrame(summary_rows)

Row,property,n,nonfinite,min,median,max
,Symbol,Int64,Int64,Float64,Float64,Float64
1,area,1834,0,2.0,251.0,24320.0
2,perimeter,1834,0,0.0,66.7696,875.436
3,convex_area,1834,15,5.0,265.0,27312.0
4,solidity,1834,15,0.591241,0.961353,1.0
5,major_axis_length,1834,0,2.0,23.7123,286.561
6,minor_axis_length,1834,0,0.0,14.4461,125.54


## 3. Timing

`\$`-interpolate the inputs, or you measure a global-variable lookup instead of
the call. Each expression is warmed up once first so no compilation lands in a
sample.

In [7]:
prop = :convex_area          # <- change me
props_one = [prop]
regionprops_table(labels; properties=props_one, convex_area_algorithm=ALG)   # warmup
@benchmark regionprops_table($labels; properties=$props_one,
                       convex_area_algorithm=$ALG) evals = 1 samples = 5 seconds = 120

BenchmarkTools.Trial: 5 samples with 1 evaluation per sample.
 Range (min … max):  2.515 s …    2.892 s  ┊ GC (min … max): 13.70% … 12.79%
 Time  (median):     2.773 s               ┊ GC (median):    12.60%
 Time  (mean ± σ):   2.725 s ± 148.723 ms  ┊ GC (mean ± σ):  13.03% ±  0.46%

  █                  █                   █    █            █  
  █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  2.51 s         Histogram: frequency by time         2.89 s <

 Memory estimate: 3.61 GiB, allocs estimate: 121889773.

In [8]:
# Every property in isolation, plus the whole set in one call.
SAMPLES, SECONDS = 5, 120

timings = DataFrame(property = String[], min_s = Float64[], median_s = Float64[],
                    alloc_mib = Float64[])

for p in PROPERTIES
    ps = [p]
    regionprops_table(labels; properties=ps, convex_area_algorithm=ALG)
    t = @benchmark regionprops_table($labels; properties=$ps,
                               convex_area_algorithm=$ALG) evals = 1 samples = SAMPLES seconds = SECONDS
    push!(timings, (string(p), minimum(t).time/1e9, BenchmarkTools.median(t).time/1e9,
                    BenchmarkTools.median(t).memory/1024^2))
end

regionprops_table(labels; properties=PROPERTIES, convex_area_algorithm=ALG)
tall = @benchmark regionprops_table($labels; properties=$PROPERTIES,
                              convex_area_algorithm=$ALG) evals = 1 samples = SAMPLES seconds = SECONDS
push!(timings, ("all_properties", minimum(tall).time/1e9,
                BenchmarkTools.median(tall).time/1e9,
                BenchmarkTools.median(tall).memory/1024^2))
timings

Row,property,min_s,median_s,alloc_mib
,String,Float64,Float64,Float64
1,area,0.121279,0.121639,0.17942
2,perimeter,2.47566,2.57743,3804.16
3,convex_area,2.42992,2.63993,3691.62
4,solidity,2.57911,2.65027,3691.63
5,major_axis_length,0.448553,0.825347,872.367
6,minor_axis_length,0.456363,0.726241,872.367
7,all_properties,3.88218,4.33564,4788.04


### Why the per-property rows do not sum to `all_properties`

`regionprops_table` computes the label lengths, the bounding boxes and the image
moments **once** and shares them across properties. Six isolated calls repeat
that work six times; one combined call pays it once. The combined figure is
what a real caller experiences.

In [9]:
singles = sum(timings.min_s[1:end-1])
combined = timings.min_s[end]
@printf("sum of isolated calls : %.4f s\n", singles)
@printf("one combined call     : %.4f s\n", combined)
@printf("shared work saves     : %.4f s  (%.1f%% of the sum)\n",
        singles - combined, 100 * (singles - combined) / singles)

sum of isolated calls : 8.5109 s
one combined call     : 3.8822 s


shared work saves     : 4.6287 s  (54.4% of the sum)


## 4. Per-floe drill-down

Bucket by area — cost and accuracy both scale with floe size, so a single
average over a scene with a heavy-tailed size distribution hides more than it
shows.

In [10]:
buckets = [(0, 100), (100, 300), (300, 1000), (1000, 5000), (5000, typemax(Int))]
rows = map(buckets) do (lo, hi)
    sel = props[(props.area .> lo) .& (props.area .<= hi), :]
    sol = filter(isfinite, sel.solidity)
    (bucket = "$lo-$(hi == typemax(Int) ? "inf" : hi)",
     n = nrow(sel),
     median_area = isempty(sel.area) ? NaN : sort(sel.area)[cld(nrow(sel), 2)],
     median_solidity = isempty(sol) ? NaN : sort(sol)[cld(length(sol), 2)],
     nan_solidity = nrow(sel) - length(sol))
end
DataFrame(rows)

Row,bucket,n,median_area,median_solidity,nan_solidity
,String,Int64,Int64,Float64,Int64
1,0-100,39,28,1.0,15
2,100-300,1018,161,0.97619,0
3,300-1000,505,454,0.94825,0
4,1000-5000,233,1495,0.912875,0
5,5000-inf,39,6717,0.887837,0


In [11]:
# The largest floes, for eyeballing individual values.
sort(props, :area; rev=true) |> df -> first(df, 10)

Row,area,convex_area,label,major_axis_length,minor_axis_length,perimeter,solidity
,Int64,Float64,Int64,Any,Any,Float64,Float64
1,24320,27312.0,348,286.561,113.02,875.436,0.890451
2,18014,20481.0,47,202.344,122.819,802.192,0.879547
3,16952,19259.0,95,278.612,81.2925,817.85,0.880212
4,15931,18110.0,750,178.297,125.54,720.654,0.87968
5,14539,15166.0,1152,232.967,80.8951,568.073,0.958658
6,13779,14498.0,1107,175.327,104.348,514.416,0.950407
7,13321,18696.0,66,192.13,120.065,657.522,0.712505
8,12311,13885.0,390,178.784,93.8471,590.541,0.88664
9,11478,13169.0,2,162.246,105.641,652.002,0.871592


## 5. Next

`02-comparison.ipynb` (Python kernel) runs the scikit-image side, joins both
result sets into the parity and speed tables, and explores the one property
that disagrees — `convex_area`, and `solidity` through it.

To regenerate the committed results rather than explore interactively:

```
OMP_NUM_THREADS=1 julia --project=benchmarks -t 1 \
    benchmarks/benchmark_regionprops.jl --samples 5
```